In [23]:
import torch
import numpy as np
# https://blog.csdn.net/bit452/article/details/109686936

- 前一部分叫做Feature Extraction，后一部分叫做classification<br>
- 每一个卷积核它的通道数量要求和输入通道是一样的。这种卷积核的总数有多少个和你输出通道的数量是一样的。<br>
- 卷积(convolution)后，C(Channels)变，W(width)和H(Height)可变可不变，取决于是否padding。subsampling(或pooling)后，C不变，W和H变。<br>
- 卷积层：保留图像的空间信息。<br>
- 卷积层要求输入输出是四维张量(B,C,W,H)，全连接层的输入与输出都是二维张量(B,Input_feature)。<br>
- 卷积(线性变换)，激活函数(非线性变换)，池化；这个过程若干次后，view打平，进入全连接层

In [24]:
import torch

in_channels, out_channels = 5, 10 # 初始通道数，要转为的通道数
width, height = 100, 100 # 图像的宽度和高度
kernel_size = 3 # 卷积核大小 3 * 3
batch_size = 1 # 批处理数

input = torch.randn(batch_size, in_channels, width, height) # bs * C * W * H
conv_layer = torch.nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size) # in_C ==> out_C 更改WH */
output = conv_layer(input) # 输出处理的图像
print(input.shape) # 大小
print(output.shape)
print(conv_layer.weight.shape) # 权重 out_C in_C ksize ksize

torch.Size([1, 5, 100, 100])
torch.Size([1, 10, 98, 98])
torch.Size([10, 5, 3, 3])


In [25]:
import torch

input = [
    3,4,6,5,7,
    2,4,6,8,2,
    1,6,7,8,4,
    9,7,4,6,2,
    3,7,5,4,1
]

input = torch.Tensor(input).view(1, 1, 5, 5) # B C W H
conv_layer = torch.nn.Conv2d(1,1, kernel_size=3, padding=1, bias = False) # padding 理解： 前端的padding
kernel = torch.Tensor([1,2,3,4,5,6,7,8,9]).view(1,1,3,3) # B C W H
conv_layer.weight.data = kernel.data # 数据赋值

output = conv_layer(input)
print(output.shape)


torch.Size([1, 1, 5, 5])


In [26]:
# Max Pooling Layer

import torch 
input = [
    3,4,6,5,
    2,4,6,8,
    1,6,7,8,
    9,7,4,6
]
input = torch.Tensor(input).view(1, 1, 4, 4)
maxpooling_layer = torch.nn.MaxPool2d(kernel_size=2) # 每2 * 2个方块内找最大值
output = maxpooling_layer(input)
print(output)

tensor([[[[4., 8.],
          [9., 8.]]]])


In [27]:
import torch
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F 
import torch.optim as optim

# prepare dataset
batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize( (0.1307, ), (0.3081, ))
])
train_dataset = datasets.MNIST(root = "../dataset/mnist/", train = True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_dataset = datasets.MNIST(root = "../dataset/mnist/", train = False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)

# design model using class

class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = torch.nn.Conv2d(1, 10, kernel_size=5) # 卷积核大小为 5 * 5
        self.conv2 = torch.nn.Conv2d(10, 20, kernel_size=5)
        self.pooling = torch.nn.MaxPool2d(2)
        self.fc = torch.nn.Linear(320, 10)

    def forward(self, x):
        batch_size = x.size(0)
        x = F.relu(self.pooling( self.conv1(x)))
        x = F.relu(self.pooling( self.conv2(x)))
        x = x.view(batch_size, -1) # -1 自动计算是320
        x = self.fc(x)
        return x
    
model = Net()

# construct loss and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.01, momentum=0.5)

# training cycle
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, target = data
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print("[ %d, %5d] loss: %.3f" % (epoch + 1, batch_idx + 1, running_loss / 300))
            running_loss = 0.0

def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim = 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print("Accuracy on test set: %d %%" % (100 * correct / total))

if __name__ == '__main__':
    for epoch in range(10):
        train(epoch)
        test()

[ 1,   300] loss: 0.664
[ 1,   600] loss: 0.195
[ 1,   900] loss: 0.143
Accuracy on test set: 96 %
[ 2,   300] loss: 0.113
[ 2,   600] loss: 0.098
[ 2,   900] loss: 0.091
Accuracy on test set: 97 %
[ 3,   300] loss: 0.072
[ 3,   600] loss: 0.076
[ 3,   900] loss: 0.071
Accuracy on test set: 98 %
[ 4,   300] loss: 0.063
[ 4,   600] loss: 0.061
[ 4,   900] loss: 0.062
Accuracy on test set: 98 %
[ 5,   300] loss: 0.053
[ 5,   600] loss: 0.056
[ 5,   900] loss: 0.054
Accuracy on test set: 98 %
[ 6,   300] loss: 0.050
[ 6,   600] loss: 0.050
[ 6,   900] loss: 0.049
Accuracy on test set: 98 %
[ 7,   300] loss: 0.044
[ 7,   600] loss: 0.047
[ 7,   900] loss: 0.041
Accuracy on test set: 98 %
[ 8,   300] loss: 0.037
[ 8,   600] loss: 0.046
[ 8,   900] loss: 0.041
Accuracy on test set: 98 %
[ 9,   300] loss: 0.034
[ 9,   600] loss: 0.039
[ 9,   900] loss: 0.041
Accuracy on test set: 98 %
[ 10,   300] loss: 0.034
[ 10,   600] loss: 0.036
[ 10,   900] loss: 0.037
Accuracy on test set: 98 %
